In [1]:
import sys
from pathlib import Path
sys.path[:0] = [str(Path.cwd().parent)]

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from self_consistency_k import bdg_sc_full_k

In [2]:
corr_strings = [
    "F_onsite",
    "F_swave",
    "F_dwave",
    "F_px",
    "F_py",
    "Fuu_px",
    "Fuu_py",
    "Fdd_px",
    "Fdd_py",
]

In [3]:
def stable_config(out, atol=1e-6, rtol=0.1):
    stable = {}
    F_max = out.corr[np.argmax(np.abs(out.corr))]
    if np.abs(F_max) < atol:
        return stable
    for i, c in enumerate(out[1:-3]):
        if np.abs(c) > atol and np.abs(c) > rtol * np.abs(F_max):
            stable[corr_strings[i]] = c
    return stable

In [4]:
def same_stable_dict(d1, d2, atol=1e-6, rtol=1e-3):
    if d1.keys() != d2.keys():
        return False

    for k in d1:
        if not np.isclose(d1[k], d2[k], atol=atol, rtol=rtol):
            return False

    return True

In [5]:
initial_seeds = np.array([
    [0,0,0,0],  # normal state
    [0.1, -0.1, 0, 0],  # px 
    [0, 0, 0.1, -0.1],  # py
    [0.1, -0.1, 0.1, -0.1],  # px + py
    [0.1, -0.1, 0.1j, -0.1j],  # px + i*py
    # [0.1, 0.1, -0.1, -0.1],  # d-wave
    # [0.1+0.1, 0.1-0.1, -0.1, -0.1],  # d-wave + px
    # [0.1, 0.1, -0.1+0.1, -0.1-0.1],  # d-wave + py
    # [0.1, 0.1, 0.1, 0.1],  # s-wave
    # [0.1+0.1, 0.1-0.1, 0.1, 0.1],  # s-wave + px
    # [0.1, 0.1, 0.1+0.1, 0.1-0.1] # s-wave + py
], dtype=np.complex128)
seed_strings = [
    "normal state",
    "px", "py", "px+py", "px+i*py",
    # "d-wave", "d-wave+px", "d-wave+py",
    # "s-wave", "s-wave+px", "s-wave+py"
]

In [18]:
mu_arr = np.linspace(1.8, 4.0, 1)
T_arr = np.linspace(0.001, 0.2, 1)

mu_arr = [1.8]
T_arr = [0.05, 0.2, 0.3, 0.4]
free_tol = 0.01
print(mu_arr)
print(T_arr)

[1.8]
[0.05, 0.2, 0.3, 0.4]


In [19]:
t=1
V_prime=1.5

h = np.array([0, 0, 0])
Nx, Ny = 30, 30

atol = 1e-6
rtol = 1e-3
maxiter=1000

In [20]:
print_output = True
records = []

mu = mu_arr[0]

for T in T_arr:
    best_free = np.inf
    configs = []
    print("====================================")
    print(f"mu={mu:.2f}, T={T:.4f}")
    print("====================================")
    for seed1, seed_str1 in zip(initial_seeds, seed_strings):
        for seed2, seed_str2 in zip(initial_seeds, seed_strings):
            print(f"  Seed-up: {seed_str1}")
            print(f"  Seed-down: {seed_str2}")

            out = bdg_sc_full_k(
                t, mu, Nx, Ny, V_prime=V_prime,
                temperature=T, maxiter=maxiter,
                atol=atol,rtol=rtol,
                Fuu_init=seed1,
                Fdd_init=seed2
            )
            if print_output == True:
                print(f"Fuu_px: {out.Fuu_px}, Fuu_py: {out.Fuu_py}")
                print(f"Fdd_px: {out.Fdd_px}, Fdd_py: {out.Fdd_py}")
                print(f"Free_energy = {out.free_energy}")
            stable = stable_config(out, atol=atol)

            print("------------------------------------------")
            if (out.free_energy < best_free 
                and np.abs(out.free_energy - best_free) > free_tol):

                best_free = out.free_energy
                configs = [{
                    "stable": stable,
                    "free": out.free_energy,
                }]

            elif np.abs(out.free_energy - best_free) <= free_tol:
                if len(stable) != 0 and not any(
                    same_stable_dict(c["stable"], stable, atol=atol, rtol=rtol)
                    for c in configs
                ):
                    configs.append({
                        "stable": stable,
                        "free": out.free_energy,
                    })

            print(configs)
            print("------------------------------------------")

    # Append one row per (mu, T)
    records.append({
        "mu": mu,
        "T": T,
        "best_free": best_free,
        "configs": configs
    })

# Create DataFrame
df = pd.DataFrame(records)

mu=1.80, T=0.0500
  Seed-up: normal state
  Seed-down: normal state
Fuu_px: 0j, Fuu_py: 0j
Fdd_px: 0j, Fdd_py: 0j
Free_energy = -2006.500241284609
------------------------------------------
[{'stable': {}, 'free': np.float64(-2006.500241284609)}]
------------------------------------------
  Seed-up: normal state
  Seed-down: px
Fuu_px: 0j, Fuu_py: 0j
Fdd_px: (0.019041355440988977+2.485906429029695e-14j), Fdd_py: (2.4870208702783975e-14-0.019040404243482974j)
Free_energy = -2007.3799951814863
------------------------------------------
[{'stable': {'Fdd_px': np.complex128(0.019041355440988977+2.485906429029695e-14j), 'Fdd_py': np.complex128(2.4870208702783975e-14-0.019040404243482974j)}, 'free': np.float64(-2007.3799951814863)}]
------------------------------------------
  Seed-up: normal state
  Seed-down: py
Fuu_px: 0j, Fuu_py: 0j
Fdd_px: (2.3037545549597078e-11-0.019040339920407406j), Fdd_py: (0.019041419713591072+2.3030919683495092e-11j)
Free_energy = -2007.3799951787103
------------

In [21]:
print(df[0:100])

    mu     T    best_free                                            configs
0  1.8  0.05 -2008.259749  [{'stable': {'Fuu_px': (0.019041242929364856+2...
1  1.8  0.20 -2020.466877  [{'stable': {'Fuu_px': (0.019040987632318932+2...
2  1.8  0.30 -2037.619325  [{'stable': {'Fuu_px': (0.0190409118847054+2.4...
3  1.8  0.40 -2062.271235  [{'stable': {'Fuu_px': (0.019040889412084185+2...


In [22]:
def flatten_df(df):
    df_flat = df.explode("configs").reset_index(drop=True)

    # Split out the free energy and stable dict
    df_flat["free"] = df_flat["configs"].apply(
        lambda x: x.get("free") if isinstance(x, dict) else 0
    )
    df_flat["stable"] = df_flat["configs"].apply(
        lambda x: x.get("stable") if isinstance(x, dict) else {}
    )

    # Expand the stable dictionaries into columns
    stable_df = pd.DataFrame(df_flat["stable"].tolist())

    # Ensure all desired columns exist
    stable_df = stable_df.reindex(columns=corr_strings)

    # Combine everything
    df_flat = pd.concat(
        [
            df_flat.drop(columns=["configs", "stable"]),
            stable_df
        ],
        axis=1
    )
    df_flat.fillna(0, inplace=True)

    return df_flat

In [25]:
df_flat = flatten_df(df)

df_flat.iloc[0:50]

,mu,T,best_free,free,F_onsite,F_swave,F_dwave,F_px,F_py,Fuu_px,Fuu_py,Fdd_px,Fdd_py
0,1.8,0.05,-2008.259749,-2008.259749,0.0,0.0,0.0,0.0,0.0,0.019041+0.000000j,0.000000-0.019041j,0.019041+0.000000j,0.000000-0.019041j
1,1.8,0.05,-2008.259749,-2008.259749,0.0,0.0,0.0,0.0,0.0,0.019041+0.000000j,0.000000-0.019041j,0.000000-0.019040j,0.019041+0.000000j
2,1.8,0.05,-2008.259749,-2008.259751,0.0,0.0,0.0,0.0,0.0,0.019041+0.000000j,0.000000-0.019041j,0.019041-0.000000j,0.000000+0.019041j
3,1.8,0.05,-2008.259749,-2008.259749,0.0,0.0,0.0,0.0,0.0,0.000000-0.019041j,0.019041+0.000000j,0.019041+0.000000j,0.000000-0.019041j
4,1.8,0.05,-2008.259749,-2008.259749,0.0,0.0,0.0,0.0,0.0,0.000000-0.019041j,0.019041+0.000000j,0.000000-0.019041j,0.019041+0.000000j
5,1.8,0.05,-2008.259749,-2008.259750,0.0,0.0,0.0,0.0,0.0,0.000000-0.019041j,0.019041+0.000000j,0.019041-0.000000j,0.000000+0.019041j
6,1.8,0.05,-2008.259749,-2008.259750,0.0,0.0,0.0,0.0,0.0,0.019041-0.000000j,0.000000+0.019041j,0.019041+0.000000j,0.000000-0.019041j
7,1.8,0.05,-2008.259749,-2008.259750,0.0,0.0,0.0,0.0,0.0,0.019041-0.000000j,0.000000+0.019041j,0.000000-0.019041j,0.019041+0.000000j
8,1.8,0.05,-2008.259749,-2008.259750,0.0,0.0,0.0,0.0,0.0,0.019041-0.000000j,0.000000+0.019041j,0.019041-0.000000j,0.000000+0.019041j
9,1.8,0.20,-2020.466877,-2020.466877,0.0,0.0,0.0,0.0,0.0,0.019041+0.000000j,0.000000-0.019041j,0.019041+0.000000j,0.000000-0.019041j


In [24]:
# df_flat.to_parquet("results_flat.parquet", index=False)
df_flat.to_json("results.json")